# Resume Matching System - Demo & Evaluation

This notebook demonstrates the AI-powered resume matching system and evaluates its performance on a synthetic dataset.

## Overview

- **Primary Approach**: Semantic embeddings using Sentence Transformers (`all-mpnet-base-v2`)
- **Baseline**: TF-IDF for comparison
- **Evaluation**: 10 sample resumes with manual labels

In [3]:
# Setup imports
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from preprocessing import load_text_file, preprocess_for_matching, load_resumes_from_directory
from tfidf_matcher import TFIDFMatcher
from embeddings import EmbeddingMatcher
from matcher import ResumeMatcher
from evaluation import (
    load_evaluation_labels,
    extract_ground_truth,
    evaluate_matcher,
    compare_methods,
    precision_at_k,
    recall_at_k,
    ndcg_at_k,
    spearman_correlation,
)

print("All imports successful!")

All imports successful!


## 1. Load the Job Description

In [4]:
# Load job description
jd_path = Path.cwd().parent / 'data' / 'job_description.txt'
job_description = load_text_file(jd_path)

print("JOB DESCRIPTION")
print("=" * 60)
print(job_description[:1500] + "..." if len(job_description) > 1500 else job_description)

JOB DESCRIPTION
Backend Software Engineer

Company: TechCorp Inc.
Location: San Francisco, CA (Remote OK)
Employment Type: Full-time

About the Role:
We are looking for an experienced Backend Software Engineer to join our growing engineering team. You will be responsible for designing, developing, and maintaining scalable backend services that power our core platform.

Responsibilities:
- Design and implement RESTful APIs and microservices
- Write clean, maintainable, and well-tested code
- Collaborate with frontend engineers and product managers
- Optimize application performance and scalability
- Participate in code reviews and technical discussions
- Troubleshoot and debug production issues
- Contribute to system architecture decisions

Required Qualifications:
- 3+ years of experience in backend software development
- Strong proficiency in Python, Java, or Go
- Experience with relational databases (PostgreSQL, MySQL)
- Familiarity with RESTful API design principles
- Understanding 

## 2. Load the Evaluation Dataset

In [5]:
# Load evaluation labels
labels_path = Path.cwd().parent / 'data' / 'evaluation_labels.json'
eval_data = load_evaluation_labels(labels_path)

# Convert to DataFrame for better visualization
labels_df = pd.DataFrame(eval_data['evaluations'])
labels_df = labels_df[['resume_file', 'category', 'label', 'reasoning']]

print(f"\nLoaded {len(labels_df)} labeled resumes")
print("\nLabel Distribution:")
print(labels_df['category'].value_counts())

labels_df[['resume_file', 'category', 'label']]


Loaded 10 labeled resumes

Label Distribution:
category
poor_match       4
good_match       3
partial_match    3
Name: count, dtype: int64


,resume_file,category,label
0,resume_01_backend_senior.txt,good_match,1.0
1,resume_02_backend_mid.txt,good_match,1.0
2,resume_03_backend_go.txt,good_match,1.0
3,resume_04_frontend.txt,partial_match,0.5
4,resume_05_devops.txt,partial_match,0.5
5,resume_06_data_engineer.txt,partial_match,0.5
6,resume_07_accountant.txt,poor_match,0.0
7,resume_08_nurse.txt,poor_match,0.0
8,resume_09_teacher.txt,poor_match,0.0
9,resume_10_marketing.txt,poor_match,0.0


## 3. Initialize the Resume Matcher

We'll use our `ResumeMatcher` class which combines both embedding and TF-IDF approaches.

In [7]:
# Initialize matcher with the high-quality model
print("Initializing matcher (this may take a moment to download the model)...")
matcher = ResumeMatcher(embedding_model='all-mpnet-base-v2')

# Load job description and resumes
matcher.load_job_description(str(jd_path))
matcher.load_resumes(str(Path.cwd().parent / 'data' / 'resumes'))

print(f"Loaded {len(matcher.resumes_raw)} resumes")
print("\nResume files:")
for name in sorted(matcher.resumes_raw.keys()):
    print(f"  - {name}")

Initializing matcher (this may take a moment to download the model)...
Loaded 10 resumes

Resume files:
  - resume_01_backend_senior.txt
  - resume_02_backend_mid.txt
  - resume_03_backend_go.txt
  - resume_04_frontend.txt
  - resume_05_devops.txt
  - resume_06_data_engineer.txt
  - resume_07_accountant.txt
  - resume_08_nurse.txt
  - resume_09_teacher.txt
  - resume_10_marketing.txt


## 4. Run Matching and View Results

In [8]:
# Get detailed results
print("Running matching algorithms...")
results = matcher.get_detailed_results()

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Merge with ground truth labels
ground_truth = extract_ground_truth(eval_data)
results_df['true_label'] = results_df['resume_id'].map(ground_truth)
results_df['category'] = results_df['true_label'].map({1.0: 'Good Match', 0.5: 'Partial Match', 0.0: 'Poor Match'})

print("\n" + "="*80)
print("MATCHING RESULTS (sorted by embedding score)")
print("="*80)
results_df[['resume_id', 'embedding_score', 'tfidf_score', 'true_label', 'category']]

Running matching algorithms...

MATCHING RESULTS (sorted by embedding score)


,resume_id,embedding_score,tfidf_score,true_label,category
0,resume_02_backend_mid.txt,0.7685,0.1194,1.0,Good Match
1,resume_03_backend_go.txt,0.7564,0.1588,1.0,Good Match
2,resume_01_backend_senior.txt,0.7551,0.1727,1.0,Good Match
3,resume_05_devops.txt,0.6622,0.0692,0.5,Partial Match
4,resume_06_data_engineer.txt,0.5924,0.0638,0.5,Partial Match
5,resume_04_frontend.txt,0.5882,0.0449,0.5,Partial Match
6,resume_10_marketing.txt,0.3540,0.0148,0.0,Poor Match
7,resume_08_nurse.txt,0.2434,0.0072,0.0,Poor Match
8,resume_07_accountant.txt,0.2302,0.0140,0.0,Poor Match
9,resume_09_teacher.txt,0.1544,0.0056,0.0,Poor Match


## 5. Compare Rankings: Embedding vs TF-IDF

In [9]:
# Get rankings
embedding_ranked = matcher.rank_resumes('embedding')
tfidf_ranked = matcher.rank_resumes('tfidf')

# Create comparison DataFrame
comparison_data = []
for i, ((emb_id, emb_score), (tfidf_id, tfidf_score)) in enumerate(zip(embedding_ranked, tfidf_ranked), 1):
    comparison_data.append({
        'Rank': i,
        'Embedding Resume': emb_id.replace('.txt', '').replace('resume_', ''),
        'Embedding Score': f"{emb_score:.4f}",
        'Embedding Label': ground_truth.get(emb_id, '?'),
        'TF-IDF Resume': tfidf_id.replace('.txt', '').replace('resume_', ''),
        'TF-IDF Score': f"{tfidf_score:.4f}",
        'TF-IDF Label': ground_truth.get(tfidf_id, '?'),
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nSide-by-side Ranking Comparison")
print("="*80)
comparison_df


Side-by-side Ranking Comparison


,Rank,Embedding Resume,Embedding Score,Embedding Label,TF-IDF Resume,TF-IDF Score,TF-IDF Label
0,1,02_backend_mid,0.7685,1.0,01_backend_senior,0.1727,1.0
1,2,03_backend_go,0.7564,1.0,03_backend_go,0.1588,1.0
2,3,01_backend_senior,0.7551,1.0,02_backend_mid,0.1194,1.0
3,4,05_devops,0.6622,0.5,05_devops,0.0692,0.5
4,5,06_data_engineer,0.5924,0.5,06_data_engineer,0.0638,0.5
5,6,04_frontend,0.5882,0.5,04_frontend,0.0449,0.5
6,7,10_marketing,0.3540,0.0,10_marketing,0.0148,0.0
7,8,08_nurse,0.2434,0.0,07_accountant,0.0140,0.0
8,9,07_accountant,0.2302,0.0,08_nurse,0.0072,0.0
9,10,09_teacher,0.1544,0.0,09_teacher,0.0056,0.0


## 6. Evaluate with Metrics

Let's calculate standard information retrieval metrics.

In [10]:
# Run comparison
comparison = compare_methods(
    embedding_ranked,
    tfidf_ranked,
    ground_truth,
    k_values=[3, 5, 10]
)

# Create metrics DataFrame
metrics_data = []
for metric in ['precision@3', 'precision@5', 'recall@3', 'recall@5', 'ndcg@3', 'ndcg@5', 
               'spearman_correlation', 'mean_absolute_error']:
    if metric in comparison['embedding']:
        emb_val = comparison['embedding'][metric]
        tfidf_val = comparison['tfidf'][metric]
        improvement = comparison['improvement'].get(metric, 0)
        
        metrics_data.append({
            'Metric': metric,
            'Embedding': emb_val,
            'TF-IDF': tfidf_val,
            'Improvement (%)': f"{improvement:+.1f}%" if metric != 'mean_absolute_error' else 'N/A'
        })

metrics_df = pd.DataFrame(metrics_data)
print("\nEVALUATION METRICS")
print("="*80)
metrics_df


EVALUATION METRICS


,Metric,Embedding,TF-IDF,Improvement (%)
0,precision@3,1.0000,1.0000,+0.0%
1,precision@5,1.0000,1.0000,+0.0%
2,recall@3,0.5000,0.5000,+0.0%
3,recall@5,0.8333,0.8333,+0.0%
4,ndcg@3,1.0000,1.0000,+0.0%
5,ndcg@5,1.0000,1.0000,+0.0%
6,spearman_correlation,0.9439,0.9439,+0.0%
7,mean_absolute_error,0.2045,0.3913,N/A


## 7. Score Distribution Analysis

In [12]:
# Analyze score distributions by category
print("\nSCORE DISTRIBUTION BY CATEGORY")
print("="*80)

for category in ['Good Match', 'Partial Match', 'Poor Match']:
    cat_data = results_df[results_df['category'] == category]
    if len(cat_data) > 0:
        print(f"\n{category} (Label: {cat_data['true_label'].iloc[0]}):")
        print(f"  Embedding scores: {cat_data['embedding_score'].min():.4f} - {cat_data['embedding_score'].max():.4f} (mean: {cat_data['embedding_score'].mean():.4f})")
        print(f"  TF-IDF scores:    {cat_data['tfidf_score'].min():.4f} - {cat_data['tfidf_score'].max():.4f} (mean: {cat_data['tfidf_score'].mean():.4f})")


SCORE DISTRIBUTION BY CATEGORY

Good Match (Label: 1.0):
  Embedding scores: 0.7551 - 0.7685 (mean: 0.7600)
  TF-IDF scores:    0.1194 - 0.1727 (mean: 0.1503)

Partial Match (Label: 0.5):
  Embedding scores: 0.5882 - 0.6622 (mean: 0.6143)
  TF-IDF scores:    0.0449 - 0.0692 (mean: 0.0593)

Poor Match (Label: 0.0):
  Embedding scores: 0.1544 - 0.3540 (mean: 0.2455)
  TF-IDF scores:    0.0056 - 0.0148 (mean: 0.0104)


## 8. Demonstrating Semantic Understanding

Let's show how embeddings understand semantic similarity between different phrasings.

In [13]:
# Demonstrate semantic understanding
from embeddings import compute_similarity

# Test pairs showing semantic understanding
test_pairs = [
    ("Python developer", "Software engineer using Python"),
    ("REST API", "RESTful web services"),
    ("Machine Learning", "ML"),
    ("5 years of experience", "half decade of professional work"),
    ("Backend developer", "Frontend engineer"),  # Should be lower
    ("Software engineer", "Marketing manager"),  # Should be very low
]

print("\nSEMANTIC SIMILARITY EXAMPLES")
print("="*80)
print("These examples demonstrate how embeddings understand meaning:\n")

for text1, text2 in test_pairs:
    similarity = compute_similarity(text1, text2, model_name='all-MiniLM-L6-v2')
    print(f'"{text1}" vs "{text2}"')
    print(f"  Similarity: {similarity:.4f}")
    print()


SEMANTIC SIMILARITY EXAMPLES
These examples demonstrate how embeddings understand meaning:

"Python developer" vs "Software engineer using Python"
  Similarity: 0.8431

"REST API" vs "RESTful web services"
  Similarity: 0.7166

"Machine Learning" vs "ML"
  Similarity: 0.3727

"5 years of experience" vs "half decade of professional work"
  Similarity: 0.4865

"Backend developer" vs "Frontend engineer"
  Similarity: 0.6684

"Software engineer" vs "Marketing manager"
  Similarity: 0.4140



## 9. Summary & Key Findings

### Results Summary

In [14]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\n1. RANKING ACCURACY:")
print(f"   - Both methods correctly identified all 3 'Good Match' resumes in top 3")
print(f"   - Precision@3: {comparison['embedding']['precision@3']:.1%} (both methods)")

print("\n2. SCORE QUALITY:")
emb_mae = comparison['embedding']['mean_absolute_error']
tfidf_mae = comparison['tfidf']['mean_absolute_error']
print(f"   - Embedding MAE: {emb_mae:.4f}")
print(f"   - TF-IDF MAE:    {tfidf_mae:.4f}")
print(f"   - Embedding scores are {((tfidf_mae - emb_mae) / tfidf_mae * 100):.1f}% closer to true labels")

print("\n3. SCORE SEPARATION:")
good_mean = results_df[results_df['category'] == 'Good Match']['embedding_score'].mean()
poor_mean = results_df[results_df['category'] == 'Poor Match']['embedding_score'].mean()
print(f"   - Mean embedding score for 'Good Match': {good_mean:.4f}")
print(f"   - Mean embedding score for 'Poor Match': {poor_mean:.4f}")
print(f"   - Clear separation: {good_mean - poor_mean:.4f} difference")

print("\n4. RECOMMENDATION:")
print("   - Use embedding-based matching as primary method")
print("   - Scores provide meaningful relevance ranking")
print("   - Consider threshold ~0.5 to filter out poor matches")


SUMMARY

1. RANKING ACCURACY:
   - Both methods correctly identified all 3 'Good Match' resumes in top 3
   - Precision@3: 100.0% (both methods)

2. SCORE QUALITY:
   - Embedding MAE: 0.2045
   - TF-IDF MAE:    0.3913
   - Embedding scores are 47.7% closer to true labels

3. SCORE SEPARATION:
   - Mean embedding score for 'Good Match': 0.7600
   - Mean embedding score for 'Poor Match': 0.2455
   - Clear separation: 0.5145 difference

4. RECOMMENDATION:
   - Use embedding-based matching as primary method
   - Scores provide meaningful relevance ranking
   - Consider threshold ~0.5 to filter out poor matches


## 10. Try It Yourself

You can test the matcher with your own text:

In [15]:
# Try matching a custom resume
custom_resume = """
Senior Software Engineer with 7 years of backend development experience.
Expert in Python, Django, and PostgreSQL. Built scalable microservices
handling millions of requests daily. Experience with AWS, Docker, and Kubernetes.
Led a team of 5 engineers on critical infrastructure projects.
"""

# Score the custom resume
matcher.add_resume('custom_test', custom_resume)
custom_results = matcher.match_all()

print("\nCUSTOM RESUME TEST")
print("="*80)
print(custom_resume.strip())
print("\nScores:")
print(f"  Embedding Score: {custom_results['custom_test']['embedding_score']:.4f}")
print(f"  TF-IDF Score:    {custom_results['custom_test']['tfidf_score']:.4f}")
print(f"\nInterpretation: This is likely a {'GOOD' if custom_results['custom_test']['embedding_score'] > 0.6 else 'PARTIAL' if custom_results['custom_test']['embedding_score'] > 0.4 else 'POOR'} match!")


CUSTOM RESUME TEST
Senior Software Engineer with 7 years of backend development experience.
Expert in Python, Django, and PostgreSQL. Built scalable microservices
handling millions of requests daily. Experience with AWS, Docker, and Kubernetes.
Led a team of 5 engineers on critical infrastructure projects.

Scores:
  Embedding Score: 0.7317
  TF-IDF Score:    0.1263

Interpretation: This is likely a GOOD match!


---

## Conclusion

This demo showed:

1. **Semantic Understanding**: The embedding-based approach understands that "Python developer" and "software engineer" are related concepts

2. **Accurate Ranking**: Both methods correctly ranked good matches above poor matches

3. **Better Score Calibration**: Embedding scores (0.15-0.77) provide more meaningful relevance signals than TF-IDF (0.01-0.17)

4. **Practical Utility**: A threshold around 0.5 could effectively filter out unqualified candidates

### Next Steps for Production

- Fine-tune the model on domain-specific HR data
- Add structured extraction for years of experience, education level
- Implement explainability features to show why a resume matched
- A/B test against human recruiter rankings